# Kaggriculture Imitation Deep Reinforcement Replay Controller

## Path B — Dueling Double DQN with action branching

Multi-branch action space: farmer (15) × crop × 6 hands × 10 market order slots → **122 Q-heads** with dueling architecture.

| Phase | What runs |
|-------|-----------|
| **Bootstrap** | Daily top episode JSONs → replay buffer + streaming BC |
| **Self-play** | Online agent vs **checkpoint pool** (historical selves) |
| **Ladder eval** | Post-training head-to-head vs all 10 agents in `opponents/` (reporting only — no retrain on failure) |

| Root | Writable | Purpose |
|------|----------|---------|
| `/kaggle/input/` | **No** | Episode JSONs, code dataset, reference agents |
| `/kaggle/working/` | **Yes** | Metadata cache, checkpoints, metrics, `agent.py` |

**Training mode:** `dry_run` (local: 1 bootstrap day + self-play + ladder). Set `KAGGLE_TRAINING_MODE=medium|full` on Kaggle GPU for production.

**Fresh run:** `KAGGLE_FRESH_RUN=1` skips resume. **`dry_run` resume:** set `KAGGLE_RESUME=1` to continue from checkpoint.

**GPU:** Kaggle → Settings → Accelerator → GPU T4 x2 → Save Version → Run All.



## §0. Sync kernel from Kaggle

Pull latest `scottweeden/kaggriculture-self-training`, patch `kernel-metadata.json` with yesterday's daily episode dataset (if missing), and promote to the repo-canonical `kaggriculture-self-training/` directory (local dev only).

Run this cell locally when you want the GitHub copy to match Kaggle. Requires `kaggle` CLI credentials and internet.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import date, timedelta
from pathlib import Path

KERNEL_SLUG = "scottweeden/kaggriculture-self-training"
NOTEBOOK_NAME = "kaggriculture-self-training.ipynb"
METADATA_NAME = "kernel-metadata.json"
STAGING_DIRNAME = "scottweeden-kaggriculture-self-training"


def log(msg: str) -> None:
    print(msg, flush=True)


KAGGLE_INPUT = (
    Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("~/kagg").expanduser()
).resolve()
KAGGLE_WORKING = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("~/kagg/working").expanduser()
).resolve()
STAGING = KAGGLE_WORKING / STAGING_DIRNAME
CANONICAL = KAGGLE_INPUT / "kaggriculture-self-training"

STAGING.mkdir(parents=True, exist_ok=True)
log(f"Staging dir: {STAGING}")
log(f"Canonical dir: {CANONICAL} (exists={CANONICAL.is_dir()})")

log(f"Pulling {KERNEL_SLUG} ...")
subprocess.run(
    ["kaggle", "kernels", "pull", KERNEL_SLUG, "-p", str(STAGING), "-m"],
    check=True,
)

pulled_ipynb = STAGING / NOTEBOOK_NAME
if not pulled_ipynb.exists():
    matches = list(STAGING.glob("*.ipynb"))
    if len(matches) == 1:
        pulled_ipynb = matches[0]
    elif matches:
        pulled_ipynb = max(matches, key=lambda p: p.stat().st_mtime)
    else:
        raise FileNotFoundError(f"No notebook found under {STAGING} after kernels pull")
log(f"Pulled notebook: {pulled_ipynb}")

metadata_path = STAGING / METADATA_NAME
if metadata_path.exists():
    log(f"Using metadata from pull: {metadata_path}")
elif (CANONICAL / METADATA_NAME).exists():
    shutil.copy2(CANONICAL / METADATA_NAME, metadata_path)
    log(f"Seeded metadata from canonical: {metadata_path}")
else:
    raise FileNotFoundError(
        f"Missing {METADATA_NAME} in staging and {CANONICAL / METADATA_NAME}"
    )

meta = json.loads(metadata_path.read_text(encoding="utf-8"))
sources = list(meta.get("dataset_sources") or [])

end_date = (date.today() - timedelta(days=1)).isoformat()
new_slug = f"kaggle/kaggriculture-episodes-{end_date}"
already = any(end_date in entry for entry in sources)

if already:
    log(f"dataset_sources already contains {end_date}; no change")
else:
    episode_re = re.compile(r"^kaggle/kaggriculture-episodes-\d{4}-\d{2}-\d{2}$")
    episodes = [s for s in sources if episode_re.match(s)]
    other = [s for s in sources if s not in episodes]
    episodes.append(new_slug)
    episodes.sort()
    meta["dataset_sources"] = episodes + other
    metadata_path.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")
    log(f"Added {new_slug} to dataset_sources")

if CANONICAL.is_dir() and os.access(CANONICAL, os.W_OK):
    shutil.copy2(metadata_path, CANONICAL / METADATA_NAME)
    log(f"Promoted → {CANONICAL / METADATA_NAME}")
    can_ipynb = CANONICAL / NOTEBOOK_NAME
    if can_ipynb.exists() and can_ipynb.stat().st_mtime > pulled_ipynb.stat().st_mtime:
        log(f"Skip notebook promote: canonical newer than Kaggle pull ({can_ipynb})")
    else:
        shutil.copy2(pulled_ipynb, can_ipynb)
        log(f"Promoted → {can_ipynb}")
else:
    log(
        f"Skip promote: {CANONICAL} missing or not writable "
        "(expected on Kaggle runtime; staging copy is authoritative for this session)"
    )

log("§0 kernel sync complete")

## 1a. Deploy code


In [ ]:
import os
import shutil
import sys
from pathlib import Path

import torch

KAGGLE_INPUT = (
    Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("~/kagg").expanduser()
).resolve()
KAGGLE_WORKING = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("~/kagg/working").expanduser()
).resolve()
CODE_CANDIDATES = [
    KAGGLE_INPUT / "datasets" / "scottweeden" / "self-training-code",
    KAGGLE_INPUT / "self-training-code",
    Path("/kaggle/input/datasets/scottweeden/kaggriculture-self-training-code"),
    Path("/kaggle/input/kaggriculture-self-training-code"),
]
CODE_SRC = next((p for p in CODE_CANDIDATES if (p / "episode_catalog.py").exists()), None)
if CODE_SRC is None:
    raise FileNotFoundError("Missing scottweeden/kaggriculture-self-training-code on input path")

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
READ_ONLY = {"kaggriculture_adapter.py", "kaggriculture_path_b_rebuild.py", "notebook_paths.py"}
MODULES = [
    "kaggriculture_self_play_training.py",
    "kaggriculture_dataset_publish.py",
    "episode_catalog.py",
    "path_b_bootstrap.py",
    "kaggriculture_adapter.py",
    "kaggriculture_path_b_rebuild.py",
    "kaggle_env_wrapper.py",
    "dataset_loader.py",
    "eval_policy.py",
    "visualize.py",
    "notebook_paths.py",
]
for name in MODULES:
    if name in READ_ONLY:
        if not (CODE_SRC / name).exists():
            raise FileNotFoundError(f"Missing {CODE_SRC / name}")
        continue
    shutil.copy2(CODE_SRC / name, KAGGLE_WORKING / name)
shutil.copytree(CODE_SRC / "kaggriculture_rl", KAGGLE_WORKING / "kaggriculture_rl", dirs_exist_ok=True)

os.chdir(KAGGLE_WORKING)
sys.path.insert(0, str(KAGGLE_WORKING))
sys.path.insert(0, str(CODE_SRC))

from notebook_paths import (
    bust_stale_modules,
    experiment_dir,
    metadata_dir,
)

bust_stale_modules()

METADATA_DIR = metadata_dir(KAGGLE_WORKING)
METADATA_PATH = METADATA_DIR / "metadata.json"
EXPERIMENT_DIR = experiment_dir(KAGGLE_WORKING)

print(f"Code (input, read-only adapter): {CODE_SRC}")
print(f"Training modules copied to: {KAGGLE_WORKING}")



## 1b. Configure training


In [ ]:
from notebook_paths import bust_stale_modules, dry_run_resume_requested, fresh_run_requested

bust_stale_modules()

import json
import os
from pathlib import Path

from episode_catalog import DEFAULT_END_DATE, DEFAULT_START_DATE
from kaggriculture_dataset_publish import restore_training_artifacts_from_code_dataset
from kaggriculture_self_play_training import train_self_play
from path_b_bootstrap import merge_bootstrap_state_from_code_dataset, plan_next_bootstrap_days_from_state

TRAINING_MODE = os.environ.get("KAGGLE_TRAINING_MODE", "dry_run")
if TRAINING_MODE not in ("dry_run", "medium", "full"):
    raise ValueError(f"KAGGLE_TRAINING_MODE={TRAINING_MODE!r} invalid; use dry_run, medium, or full")

MODE_PRESETS = {
    "dry_run": {
        "bootstrap_mode": "daily_incremental",
        "bootstrap_days_per_run": 1,
        "bootstrap_episodes": None,
        "bootstrap_top_per_day": None,
        "bootstrap_passes": 1,
        "bootstrap_transitions": None,
        "buffer_capacity": 50_000,
        "bc_epochs_per_pass": 1,
        "bc_steps_per_epoch": 100,
        "total_episodes": 10,
        "learning_start_episodes": 1,
        "n_eval_episodes": 2,
        "ladder_eval_episodes": 3,
        "min_self_play_episodes": 3,
        "max_episode_steps": 720,
        "_eta": "~15–30 minutes locally (720-step seasons + ladder)",
    },
    "medium": {
        "bootstrap_mode": "daily_incremental",
        "bootstrap_days_per_run": 5,
        "bootstrap_episodes": None,
        "bootstrap_top_per_day": None,
        "bootstrap_passes": 1,
        "bootstrap_transitions": None,
        "buffer_capacity": 200_000,
        "bc_epochs_per_pass": 1,
        "bc_steps_per_epoch": None,
        "total_episodes": 25,
        "learning_start_episodes": 5,
        "n_eval_episodes": 10,
        "ladder_eval_episodes": 10,
        "min_self_play_episodes": 0,
        "max_episode_steps": 720,
        "_eta": "~2–3 hours per 5-day batch",
    },
    "full": {
        "bootstrap_mode": "daily_incremental",
        "bootstrap_days_per_run": 5,
        "bootstrap_episodes": None,
        "bootstrap_top_per_day": None,
        "bootstrap_passes": 1,
        "bootstrap_transitions": None,
        "buffer_capacity": 600_000,
        "bc_epochs_per_pass": 2,
        "bc_steps_per_epoch": None,
        "total_episodes": 100,
        "learning_start_episodes": 5,
        "n_eval_episodes": 20,
        "ladder_eval_episodes": 20,
        "min_self_play_episodes": 0,
        "max_episode_steps": 720,
        "_eta": "several hours per 5-day batch",
    },
}

_mode = MODE_PRESETS[TRAINING_MODE]
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

_restored = restore_training_artifacts_from_code_dataset(CODE_SRC, EXPERIMENT_DIR)
_bootstrap_state = merge_bootstrap_state_from_code_dataset(CODE_SRC, EXPERIMENT_DIR)
if _restored:
    print(f"Restored from code dataset training_artifacts/: {_restored}")
_done = list(_bootstrap_state.get("bootstrapped_dates", []))
if _done:
    print(f"Bootstrap resume: {len(_done)} day(s) already complete through {_done[-1]}")

_next_days = plan_next_bootstrap_days_from_state(
    _done,
    n_days=_mode["bootstrap_days_per_run"],
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
)
if TRAINING_MODE == "dry_run":
    _next_days = _next_days[:1]

_resume_ckpt = EXPERIMENT_DIR / "checkpoints" / "training_state_latest.pt"


def _read_last_completed_episode(exp_dir: Path) -> int:
    cfg_path = exp_dir / "config.json"
    if cfg_path.exists():
        try:
            return int(json.loads(cfg_path.read_text()).get("last_completed_episode", 0))
        except (json.JSONDecodeError, TypeError, ValueError):
            pass
    return 0


from eval_policy import count_reference_opponent_files, resolve_opponents_dir

OPPONENTS_DIR = resolve_opponents_dir(code_src=str(CODE_SRC))
if OPPONENTS_DIR is None:
    OPPONENTS_DIR = Path(os.environ.get("KAGGLE_OPPONENTS_DIR", "~/kagg/opponents")).expanduser().resolve()

_fresh = fresh_run_requested()
_can_resume = (
    not _fresh
    and (_resume_ckpt.exists() or bool(_restored))
    and (TRAINING_MODE != "dry_run" or dry_run_resume_requested())
)
_last_ep = _read_last_completed_episode(EXPERIMENT_DIR) if _can_resume else 0
_min_new = _mode.get("min_self_play_episodes", 0)
_total_episodes = max(_mode["total_episodes"], _last_ep + _min_new)

TRAINING_CONFIG = {
    "experiment_dir": str(EXPERIMENT_DIR),
    "code_src": str(CODE_SRC),
    "use_kaggle_env": True,
    "bootstrap_mode": _mode["bootstrap_mode"],
    "bootstrap_days_per_run": 1 if TRAINING_MODE == "dry_run" else _mode["bootstrap_days_per_run"],
    "bootstrap_episodes": _mode["bootstrap_episodes"],
    "bootstrap_top_per_day": _mode["bootstrap_top_per_day"],
    "bootstrap_passes": _mode["bootstrap_passes"],
    "bootstrap_transitions": _mode["bootstrap_transitions"],
    "buffer_capacity": _mode["buffer_capacity"],
    "bc_epochs": 0,
    "bc_epochs_per_pass": _mode["bc_epochs_per_pass"],
    "bc_steps_per_epoch": _mode["bc_steps_per_epoch"],
    "data_dir": str(METADATA_DIR),
    "metadata_path": str(METADATA_PATH),
    "download_bootstrap": False,
    "bc_batch_size": 64,
    "total_episodes": _total_episodes,
    "learning_start_episodes": _mode["learning_start_episodes"],
    "batch_size": 32,
    "checkpoint_interval": 10,
    "n_eval_episodes": _mode["n_eval_episodes"],
    "ladder_eval_episodes": _mode.get("ladder_eval_episodes", 0),
    "ladder_win_rate_target": 0.5,
    "min_self_play_episodes": _min_new,
    "opponents_dir": str(OPPONENTS_DIR) if OPPONENTS_DIR.is_dir() else None,
    "max_episode_steps": _mode["max_episode_steps"],
    "device_name": "auto",
    "seed": 42,
    "resume": str(EXPERIMENT_DIR) if _can_resume else None,
    "publish_code_dataset": TRAINING_MODE != "dry_run",
    "verbose": True,
}

print(f"=== TRAINING_MODE={TRAINING_MODE!r} ({_mode['_eta']}) ===")
if _fresh:
    print("  KAGGLE_FRESH_RUN=1 → resume disabled")
elif TRAINING_MODE == "dry_run" and not dry_run_resume_requested():
    print("  dry_run: resume off (set KAGGLE_RESUME=1 to continue checkpoint)")
print(
    f"  Self-play: {_total_episodes} episodes (learn from ep {_mode['learning_start_episodes']}, "
    f"ladder={_mode.get('ladder_eval_episodes', 0)} ep/opponent)"
)
if _can_resume and _last_ep >= _mode["total_episodes"]:
    print(f"  Extended target → {_total_episodes} (+{_min_new} min self-play)")



## 1c. Index episodes & preflight


In [ ]:
from notebook_paths import bust_stale_modules

bust_stale_modules()

import json
import os
import subprocess

import torch
from episode_catalog import (
    DEFAULT_END_DATE,
    bootstrap_metadata_start_date,
    configure_local_datasets_root,
    ensure_daily_episode_dataset,
    ensure_kaggle_index_dataset,
    is_kaggle_runtime,
    merge_episode_metadata,
    save_metadata,
)
from eval_policy import count_reference_opponent_files, resolve_opponents_dir
from kaggriculture_adapter import any_accelerator_available, gpu_backend_diagnostics

_ladder_eps = TRAINING_CONFIG.get("ladder_eval_episodes", 0)
if _ladder_eps > 0:
    _opp = resolve_opponents_dir(TRAINING_CONFIG.get("opponents_dir"), code_src=str(CODE_SRC))
    if _opp is None or not _opp.is_dir():
        raise FileNotFoundError(
            "ladder_eval_episodes > 0 but opponents/ not found. "
            "Attach raykkretzschmar/kaggriculture-reference-agents, bundle opponents/ in the code dataset, "
            "or set KAGGLE_OPPONENTS_DIR."
        )
    _opp_count = count_reference_opponent_files(_opp)
    TRAINING_CONFIG["opponents_dir"] = str(_opp)
    OPPONENTS_DIR = _opp
    print(f"Opponents: {_opp} ({_opp_count} agents)")
    if _opp_count == 0:
        raise FileNotFoundError(f"No opponent modules under {_opp}")

print("=== GPU diagnostics ===")
_gpu = gpu_backend_diagnostics()
print(f"torch: {torch.__version__}")
print(f"Resolved training device: {_gpu['resolved_device']}")
print(f"KAGGLE_KERNEL_RUN_TYPE: {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '(unset)')}")
try:
    print(subprocess.check_output(["nvidia-smi", "-L"], text=True, stderr=subprocess.STDOUT).strip())
except Exception as exc:
    print(f"nvidia-smi: {exc}")

if not any_accelerator_available() and os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    raise RuntimeError("No GPU in Batch session — enable GPU T4 x2 on Kaggle and Run All.")

LOCAL_DATASETS = KAGGLE_INPUT / "datasets" / "kaggle"
if not is_kaggle_runtime():
    configure_local_datasets_root(LOCAL_DATASETS)
    ensure_kaggle_index_dataset(local_root=LOCAL_DATASETS)
    for _day in _next_days:
        ensure_daily_episode_dataset(_day, local_root=LOCAL_DATASETS)

METADATA_DIR.mkdir(parents=True, exist_ok=True)
_metadata_start = bootstrap_metadata_start_date(_done, default_start=DEFAULT_START_DATE)
metadata = merge_episode_metadata(data_dir=METADATA_DIR, start_date=_metadata_start, end_date=DEFAULT_END_DATE)
save_metadata(metadata, METADATA_PATH)

print(f"Episodes indexed: {metadata['total_episodes_indexed']:,}")
print(f"Next bootstrap day(s): {_next_days or '(corpus complete)'}")
print(json.dumps(TRAINING_CONFIG, indent=2))



## 2. Run training (bootstrap → BC → self-play → ladder eval)


In [ ]:
train_self_play(**TRAINING_CONFIG)


## 2b. Ladder eval

`train_self_play` (§2) writes `run/metrics/ladder_eval.json` — head-to-head vs all reference agents in `opponents/` after self-play.

Promotion gate (reporting): **win rate ≥ 50%** vs every opponent. Training does **not** re-run when the ladder fails; use `medium`/`full` for longer training.


## 3. Results summary


In [ ]:
import json

metrics_dir = EXPERIMENT_DIR / "metrics"

bc_path = metrics_dir / "bc_pretrain.json"
if bc_path.exists():
    bc = json.loads(bc_path.read_text())
    print("=== BC Pretrain ===")
    print(f"  Bootstrap transitions: {bc.get('bootstrap_transitions')}")
    print(f"  Final BC loss: {bc.get('final_loss'):.5f}")

wr_path = metrics_dir / "win_rate_eval.json"
if wr_path.exists():
    wr = json.loads(wr_path.read_text())
    print("\n=== Win-rate eval (random baseline) ===")
    print(f"  Win: {wr.get('win_rate', 0):.1%} ({wr.get('wins')}/{wr.get('n_episodes')})")

ladder_path = metrics_dir / "ladder_eval.json"
if ladder_path.exists():
    ladder = json.loads(ladder_path.read_text())
    print("\n=== Ladder eval (reference opponents) ===")
    print(f"  Beats all: {ladder.get('beats_all_opponents')}")
    cleared = sum(1 for r in ladder.get("results", {}).values() if r.get("cleared"))
    print(f"  Cleared: {cleared}/{len(ladder.get('results', {}))} opponents")
    for slug, row in ladder.get("results", {}).items():
        mark = "✓" if row.get("cleared") else "✗"
        print(
            f"  {mark} {slug:16s} {row.get('win_rate', 0):.0%} "
            f"({row.get('wins', 0)}/{row.get('n_episodes', 0)})"
        )
else:
    print("\n(no ladder_eval.json — run §2)")

for rel in ["agent.py", "models/model.pth", "checkpoints/training_state_latest.pt", "config.json", "metrics/ladder_eval.json"]:
    p = EXPERIMENT_DIR / rel
    print(f"  {'OK' if p.exists() else '--'} {rel}")



## 4. Visualize metrics


In [ ]:
%matplotlib inline

import json
from pathlib import Path

import matplotlib.pyplot as plt

from notebook_paths import ensure_sys_paths, experiment_dir, kaggle_input_root, kaggle_working_root, resolve_code_src
from visualize import MetricsVisualizer, TrainingMetricsLoader

ensure_sys_paths(kaggle_working_root(), resolve_code_src(kaggle_input_root()))
EXPERIMENT_DIR = globals().get("EXPERIMENT_DIR", experiment_dir())

PLOT_DIR = EXPERIMENT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
loader = TrainingMetricsLoader([str(EXPERIMENT_DIR)])
loader.load_all()

if loader.metrics:
    viz = MetricsVisualizer(output_dir=PLOT_DIR, figure_size=(12, 8))
    viz.print_summary(loader)
    figures = viz.plot_comparison(loader)
    if figures:
        viz.save_figures(figures, list(loader.metrics.keys()), close=False)
        for fig in figures:
            plt.figure(fig.number)
            plt.tight_layout()
            plt.show()
else:
    print(f"No metrics under {EXPERIMENT_DIR}")



## 5. Publish artifacts to code dataset


In [ ]:
import os

from kaggriculture_dataset_publish import (
    publish_training_artifacts_to_code_dataset,
    training_version_message,
)
from notebook_paths import experiment_dir

EXPERIMENT_DIR = globals().get("EXPERIMENT_DIR", experiment_dir())
TRAINING_MODE = globals().get("TRAINING_MODE", os.environ.get("KAGGLE_TRAINING_MODE", "dry_run"))

if TRAINING_MODE == "dry_run":
    print("Skipping code dataset publish (dry_run mode)")
else:
    summary = publish_training_artifacts_to_code_dataset(
        EXPERIMENT_DIR,
        version_message=training_version_message(EXPERIMENT_DIR),
    )
    print("Published to scottweeden/kaggriculture-self-training-code")
    for item in summary["copied_artifacts"]:
        print(f"  - {item}")

